In [3]:
! pip install langchain_community tikroken lanchain-openai langchainhub chromeadb langchain

ERROR: Could not find a version that satisfies the requirement tikroken (from versions: none)
ERROR: No matching distribution found for tikroken


In [5]:
import os



In [6]:
! pip install chromadb


In [46]:
import os
from pathlib import Path
from dotenv import load_dotenv

# Always use absolute path so it works regardless of kernel cwd
_env_path = Path("/Users/shibadityadeb/Desktop/RAG/.env")
loaded = load_dotenv(_env_path, override=True)
print(f".env loaded: {loaded} from {_env_path}")

LANGSMITH_API_KEY = os.getenv("LANGSMITH_API_KEY")
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")
print("ANTHROPIC key loaded:", bool(ANTHROPIC_API_KEY))

.env loaded: True from /Users/shibadityadeb/Desktop/RAG/.env
ANTHROPIC key loaded: True


In [17]:
os.environ["LANGSMITH_TRACING_V2"] = 'true'
os.environ['LANGSMITH_ENDPOINT'] = 'https://api.smith.langchain.com'

from langchain_anthropic import ChatAnthropic


In [18]:
import bs4
from langsmith import Client
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
# one-time install
# uv add sentence-transformers

from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_anthropic import ChatAnthropic





#### INDEXING ####

# Load Documents
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)

docs = loader.load()
# Split
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

splits = text_splitter.split_documents(docs)
# Embed
embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(documents=splits, embedding=embedding)

retriever = vectorstore.as_retriever()
#### RETRIEVAL and GENERATION ####

# Prompt
prompt = Client().pull_prompt("rlm/rag-prompt")

# LLM

llm = ChatAnthropic(
    model="claude-3-haiku-20240307",
    temperature=0
)
# Post-processing
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)
    # Chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)
# Question

rag_chain.invoke("What is Task Decomposition?")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7954.87it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


AuthenticationError: Error code: 401 - {'type': 'error', 'error': {'type': 'authentication_error', 'message': 'invalid x-api-key'}, 'request_id': 'req_011CYoHo17MJX1VTaafQrnET'}

In [20]:
import tiktoken
def num_tokens_from_string(straing:str,encoding_name:str)->int:
    """Returns the number of tokens in a text string."""
    encoding = tiktoken.get_encoding(encoding_name)
    num_tokens = len(encoding.encode(straing))
    return num_tokens
    

num_tokens_from_string(question, "cl100k_base")

6

Text Embedding Model

In [22]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
query_result = embedding.embed_query(question)
document = splits[0].page_content  # use the first document chunk as a sample
document_result = embedding.embed_query(document)
len(query_result), len(document_result)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8108.97it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


(384, 384)

STARTING WITH IMPLEMENTATION

INDEXING

In [24]:
from langchain_community.document_loaders import PyPDFLoader
import glob

pdf_dir = "../data/pdf"
pdf_files = glob.glob(f"{pdf_dir}/*.pdf")

pdf_docs = []
for path in pdf_files:
    loader = PyPDFLoader(path)
    pdf_docs.extend(loader.load())

print(f"Loaded {len(pdf_docs)} pages from {len(pdf_files)} PDF(s):")
for path in pdf_files:
    print(f"  - {path}")
docs=loader.load()


Loaded 47 pages from 3 PDF(s):
  - ../data/pdf/objectdetection.pdf
  - ../data/pdf/embeddings.pdf
  - ../data/pdf/attention.pdf


SPLITTER

In [27]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
splits = text_splitter.split_documents(pdf_docs)
print(f"Split into {len(splits)} chunks")

Split into 974 chunks


VECTOR STORE

In [30]:
#Index
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
vectorstore = Chroma.from_documents(documents=splits, embedding=embedding)
retriever = vectorstore.as_retriever()
retriever

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x1333109d0>, search_kwargs={})

RETREIVAL

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 1})
docs = retriever.invoke("What is the main topic of the document?")
docs 
len(docs)
#k tells number of nearby neighnbors to fetch

1

GEBERATION


In [47]:
from langchain_core.prompts import ChatPromptTemplate


template="""Answer the question based on the following context:\n\n{context}\n\nQuestion: {question}\nAnswer:"""
prompt=ChatPromptTemplate.from_template(template)
prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Answer the question based on the following context:\n\n{context}\n\nQuestion: {question}\nAnswer:'), additional_kwargs={})])

In [48]:
import os
api_key = os.environ.get("ANTHROPIC_API_KEY", "")
print(f"Using key: {api_key[:12]}... (length={len(api_key)})")

llm = ChatAnthropic(
    model="claude-3-haiku-20240307",
    temperature=0,
    api_key=api_key
)

Using key: sk-ant-api03... (length=108)


In [49]:
chain=prompt | llm

In [50]:
chain.invoke({"context": docs, "question": "What is the main topic of the document?"})

AIMessage(content='Based on the provided context, which includes metadata about a document, the main topic of the document is not explicitly stated. The metadata indicates that this is page 12 of a 15-page document, but does not provide any information about the overall topic or subject matter of the document. Without more context about the content of the full document, I cannot confidently determine the main topic. The metadata alone is insufficient to answer the question.', additional_kwargs={}, response_metadata={'id': 'msg_01FWRWCbpmAA3ZrZBW8gbyH5', 'container': None, 'model': 'claude-3-haiku-20240307', 'stop_reason': 'end_turn', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'not_available', 'input_tokens': 258, 'output_tokens': 92, 'server_tool_use': None, 'service_tier': 'standard'}, 'model_name': 'claude-3-haiku-20240307', 'model